In [14]:

import torch
from monai.config import print_config
from monai.losses import DiceLoss
from monai.metrics import DiceMetric,DiceHelper
from monai.transforms import AsDiscrete, Compose

print_config()


def print_tensor(name, t):
    print(f"{name}: {t.numpy().tolist()} shape: {tuple(t.shape)}")

MONAI version: 1.4.0
Numpy version: 1.26.4
Pytorch version: 2.5.0+cu124
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: 46a5272196a6c2590ca2589029eed8e4d56ff008
MONAI __file__: /home/<username>/Projet_Hemorragie/hemorragie-env/lib/python3.12/site-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: NOT INSTALLED or UNKNOWN VERSION.
ITK version: NOT INSTALLED or UNKNOWN VERSION.
Nibabel version: 5.3.2
scikit-image version: NOT INSTALLED or UNKNOWN VERSION.
scipy version: 1.15.2
Pillow version: 11.2.1
Tensorboard version: 2.19.0
gdown version: NOT INSTALLED or UNKNOWN VERSION.
TorchVision version: 0.20.0+cu124
tqdm version: 4.67.1
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 7.0.0
pandas version: 2.2.3
einops version: 0.8.1
transformers version: 4.57.1
mlflow version: NOT INSTALLED or UNKNOWN VERSION.
pynrrd version: NOT INSTALLED or UNKNOWN VERSION.
clearml version: NOT INSTALLED or UNKNOWN VERSION.

For de

In [11]:

# CHW
grnd = torch.zeros(1, 4, 4)
pred = torch.zeros(1, 4, 4)
print_tensor("grnd", grnd)
print_tensor("pred", pred)

# grnd= [0,0,0,0] pred= [0,0,0,0]
#       [0,1,1,0]       [0,1,1,0]
#       [0,1,1,0]       [1,1,0,0]
#       [0,0,0,0]       [0,0,0,0]
grnd[..., 1, 1] = grnd[..., 1, 2] = grnd[..., 2, 1] = grnd[..., 2, 2] = 1
pred[..., 1, 1] = pred[..., 1, 2] = pred[..., 2, 0] = pred[..., 2, 1] = 1
print_tensor("updated grnd", grnd)
print_tensor("updated pred", pred)

grnd: [[[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]] shape: (1, 4, 4)
pred: [[[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]] shape: (1, 4, 4)
updated grnd: [[[0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 1.0, 0.0], [0.0, 1.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0]]] shape: (1, 4, 4)
updated pred: [[[0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 1.0, 0.0], [1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]] shape: (1, 4, 4)


In [16]:
loss = DiceLoss()
metric = DiceMetric()

# using [None] to add batch dimension
print_tensor("loss", loss(pred[None], grnd[None]))
print_tensor("metric", metric(pred[None], grnd[None]))

loss: 0.24999964237213135 shape: ()
metric: [[0.75]] shape: (1, 1)


In [17]:
# make one hot and add batch dimension
make_2_class = Compose([AsDiscrete(to_onehot=2), lambda x: x[None]])

grnd2 = make_2_class(grnd)
pred2 = make_2_class(pred)

print_tensor("grnd2", grnd2)
print_tensor("pred2", pred2)

grnd2: [[[[1.0, 1.0, 1.0, 1.0], [1.0, 0.0, 0.0, 1.0], [1.0, 0.0, 0.0, 1.0], [1.0, 1.0, 1.0, 1.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 1.0, 0.0], [0.0, 1.0, 1.0, 0.0], [0.0, 0.0, 0.0, 0.0]]]] shape: (1, 2, 4, 4)
pred2: [[[[1.0, 1.0, 1.0, 1.0], [1.0, 0.0, 0.0, 1.0], [0.0, 0.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0]], [[0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 1.0, 0.0], [1.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]]] shape: (1, 2, 4, 4)


In [20]:
loss2 = DiceLoss(include_background=True, reduction="none")
metric2 = DiceMetric(include_background=True, reduction="none")

print_tensor("loss2", loss2(pred2, grnd2))
print_tensor("metric2", metric2(pred2, grnd2))

loss2: [[[[0.08333331346511841]], [[0.24999964237213135]]]] shape: (1, 2, 1, 1)
metric2: [[0.9166666865348816, 0.75]] shape: (1, 2)


In [21]:
loss_onehot = DiceLoss(include_background=True, to_onehot_y=True, reduction="none")

# note grnd[None] instead of grnd2
print_tensor("loss_onehot", loss_onehot(pred2, grnd[None]))

loss_onehot: [[[[0.08333331346511841]], [[0.24999964237213135]]]] shape: (1, 2, 1, 1)


In [22]:

loss3 = DiceLoss(include_background=False, reduction="none")
metric3 = DiceMetric(include_background=False, reduction="none")

print_tensor("loss3", loss3(pred2, grnd2))
print_tensor("metric3", metric3(pred2, grnd2))

loss3: [[[[0.24999964237213135]]]] shape: (1, 1, 1, 1)
metric3: [[0.75]] shape: (1, 1)


In [23]:
loss4 = DiceLoss(include_background=True, reduction="mean")
metric4 = DiceMetric(include_background=True, reduction="mean")

print_tensor("loss4", loss4(pred2, grnd2))
print_tensor("metric4", metric4(pred2, grnd2))

loss4: 0.16666647791862488 shape: ()
metric4: [[0.9166666865348816, 0.75]] shape: (1, 2)


In [24]:
mgrnd = torch.zeros(1, 4, 4)
mpred = torch.zeros(1, 4, 4)

# grnd= [0,0,0,0] pred= [0,0,0,0]
#       [0,1,1,0]       [0,1,1,0]
#       [0,2,2,0]       [2,2,0,0]
#       [0,0,0,0]       [0,0,0,0]
mgrnd[..., 1, 1] = mgrnd[..., 1, 2] = 1
mgrnd[..., 2, 1] = mgrnd[..., 2, 2] = 2

mpred[..., 1, 1] = mpred[..., 1, 2] = 1
mpred[..., 2, 0] = mpred[..., 2, 1] = 2

In [25]:
print_tensor("loss", loss(mpred[None], mgrnd[None]))
print_tensor("metric", metric(mpred[None], mgrnd[None]))

loss: 0.0 shape: ()
metric: [[1.0]] shape: (1, 1)


In [27]:
make_3_class = Compose([AsDiscrete(to_onehot=3), lambda x: x[None]])

mgrnd2 = make_3_class(mgrnd)
mpred2 = make_3_class(mpred)

print_tensor("loss", loss(mpred2, mgrnd2))
print_tensor("metric", metric(mpred2, mgrnd2))

loss: 0.19444401562213898 shape: ()
metric: [[0.9166666865348816, 1.0, 0.5]] shape: (1, 3)


In [31]:

loss5 = DiceLoss(reduction="none")
metric5 = DiceMetric(reduction="none")

print_tensor("loss5", loss5(mpred2, mgrnd2))
print_tensor("metric5", metric5(mpred2, mgrnd[None]))

loss5: [[[[0.08333331346511841]], [[0.0]], [[0.4999987483024597]]]] shape: (1, 3, 1, 1)
metric5: [[0.9166666865348816, 1.0, 0.5]] shape: (1, 3)


In [29]:

loss6 = DiceLoss(reduction="none", to_onehot_y=True)
print_tensor("loss6", loss6(mpred2, mgrnd[None]))

loss6: [[[[0.08333331346511841]], [[0.0]], [[0.4999987483024597]]]] shape: (1, 3, 1, 1)
